# Search for Overflow

## Kas Knicely, University of Alaska Fairbanks

This notebook searches Sentinel 1 SAR imagery for possible overflow locations along the Tanana River between Fairbanks and Nanana. This was created in support of the CCREL Arctic Trafficability Project. 

This was created with radiometrically terrain corrected (RTC) SAR imagery in mind, though - strictly speaking - non-RTC SAR imagery can be used as well. The non-RTC SAR imagery will only work in this tool <b>if</b> it was taken by the same sensor from the same look angle and has pixel locations that match each other. This is because a difference calculation is used to locate the possible overflow. 

To incorporate SAR imagery from MULTIPLE sensor sources (e.g., Sentinel 1 and plane-mounted), the data <b>must be</b> RTC. 

Approximate Date of Creation: 2026 Jan. 20

***
# 0. Load Python Libraries

Load necessary libraries. 

In [ ]:
# Confirmed Necessary
import xarray as xr
from pathlib import Path
import pandas as pd
import glob
import re
import os
import rasterio
import matplotlib.pyplot as plt
from matplotlib.pyplot import cm
# Possibly Necessary
import rioxarray as rxr
import time
import numpy as np
import time
from pyproj import Transformer
from datetime import date
import itertools
from scipy import signal

Install necessary libraries that are not included in the base kernel. 

In [ ]:
!pip install ipyfilechooser

In [ ]:
# Install and import skimage
!pip install scikit-image
from skimage.measure import label

In [ ]:
!pip install simplekml
import simplekml

***
# 1. User Settings

## 1.1 Select folder from which to load S1 SAR Imagery

This should be the folder containing the subsetted imagery of your region of interest. These files should be in a folder named 'RTC_GAMMA'. 

For example: <br>
Subsets were placed in a folder named 'subsets'. This folder should contain a folder named 'RTC_GAMMA' which will contain your subsetted tiffs. To run the below code, you will select the folder 'subsets'. 

In [ ]:
from ipyfilechooser import FileChooser
fc = FileChooser(Path.cwd())
display(fc)

## 1.2 Select thresholds and data to include

Please select the minimum changes in backscatter to flag a pixel as a potential location for overflow. 

min_pos - smallest positive change prior to the drop in backscatter. <br>
min_neg - smallest negative change to backscatter. 

start_ind - first index of data to include. In python, index of 0 is the first. <br>
stop_ind  - last index of data to include. Setting this to -1 will include the very last SAR acquisition.<br>

aggOrInd  - 'aggregate' or 'individual'. This is a code word which will tell later code to either get overflow for the 'aggregate' (all days between the start and stop indices) or 'individual' (for each consecutive acquisition pair between the start and stop indices). 

For example, <br>start_ind = 0 and stop_ind = -1 will include all data. <br>start_ind = 1 and stop_ind = 3 will include the 2nd, 3rd, and 4th set of data. 

In [ ]:
# set delta_dB thresholds. 
min_pos = 0.1
min_neg = [-5.0, -3.0, -2.0, -1.0]
# make sure 'min_neg' is a list
if not isinstance(min_neg, list):
    min_neg = [min_neg]
    print(f"min_neg converted to list.")
# Make sure minimum values are in order from lowest (most stringent) to highest (least stringent). 
min_neg.sort()

# Set start and stop indices. This is useful if you are dealing with e.g., several months of data that includes time that can't possibly have overflow. 
start_ind = 0
stop_ind = -1

aggOrInd = 'individual' # 'aggregate' or 'individual'

print(f"start_ind: {start_ind}\nstop_ind:  {stop_ind}")
if 'aggregate' == aggOrInd:
    print(f"Aggregate selected. This will produce one geotiff that contains all of the overflow identified between the start and stop indices.")
elif 'individual' == aggOrInd: 
    print(f"Individual selected. This will produce a geotiff for each consecutive image pair between the start and stop indices.")

Set median filter to apply to the raw data. This is often a standard step when using SAR imagery to reduce its inherently noisy nature. 

In [ ]:
median_size = 3
if median_size % 2 != 0:
    pass
else:
    print("WARNING!!!\nWARNING!!!\nWARNING!!!\nMedian filter size must be odd. Increasing value by 1.")
    median_size = median_size + 1
    print(f"median_size = {median_size}")

Select window size for overflow search. This is used to give more robust detections. The larger this value, the faster searches will proceed, but at the cost of missing smaller overflow features. 

In [ ]:
### --- Set up window sizes for search through SAR imagery --- ###

window_sizes = [3] # window sizes; must be odd value.
if not isinstance(window_sizes, list):
    window_sizes = [window_sizes]
    print("window_sizes converted to list.") 
for i in range(len(window_sizes)): 
    if window_sizes[i] % 2 != 0:
        continue
    else: 
        print("WARNING!!!\nWARNING!!!\nWARNING!!!\nWindow sizes must be odd. Increasing value by 1.")
        window_sizes[i] = window_sizes[i]+1

# remove duplicates. 
window_sizes = list(set(window_sizes))
window_sizes.sort()

## 1.3 Select minimum (and maximum) size of possible overflow locations to include

Please select the minimum number of touching pixels (min_pixels) for your overflow region. Note that the maximum may also be selected. 

In [ ]:
# Typical overflow event are approximately 15 S1 pixels in size. 
min_pixels = 2
max_pixels = float('inf')

## 1.4 Set kml and geotiff save settings

Select the location in which to save the files. This is often best set to the same thing as the load location. 

In [ ]:
fc_saveLoc = FileChooser(Path.cwd())
display(fc_saveLoc)

Select the save name for the geotiff. The default is 'overflow_[date of creation].tif'. The default name will be used when teh saveName is set to None. 

For example, <br>
saveOverflowAsGeotiff(fc, fc_saveLoc, da_VV_tot[0], myIndices_both, saveName = 'TananaRiverOverflow')

In [ ]:
saveName = None
saveName = 'multiOverflowTest'

## 1.5 Note about Section 6. 

Section 6. can display data for a specific location as identified in section 5. The default code will display the first location for the most stringent threshold applied. To change this, the user must go to section 6. and select a different location manually. 

***
# 2. Load Sentinel 1 SAR Imagery

In [ ]:
### --- Get Dates from filenames --- ###
def get_dates(flnms):
    dates = []

    for flnm in flnms:
        date_regex = r'\d{8}'
        date = re.search(date_regex, str(flnm))
        if date:
            dates.append(date.group(0))
    return dates


### --- Load Geotiffs Function --- ###
def load_tiffs(parent, type_file, pola, stop_ind = -1):

    # Load the appropriate files
    if type_file == 'SAR':
        folder = parent+'RTC_GAMMA/'
        prefix = f'*{pola}'
    else:
        folder = parent+'Water_Masks/'
        prefix = '*combined'

    # Gather names of files corresponding to the file type and polarization we want
    tiff_dir = Path(folder)
    tiffs = [f for f in os.listdir(tiff_dir) if pola in f]
    
    # Gather the date of each file
    times = get_dates(tiffs)

    # Make 'tiffs' have full path
    tiffs = [Path(folder,tiff) for tiff in tiffs]
    
    # Create a list of indices based on the sorted order of times
    sorted_indices = sorted(range(len(times)), key=lambda i: times[i])
    
    # Sort the paths based on the times
    tiffs = [tiffs[i] for i in sorted_indices]
    
    # Sort the times axisdd
    times.sort()
    times = pd.DatetimeIndex(times)
    times.name = "time"
    
    # Create the dataset gathering the input images
    if stop_ind == -1:
        stop_ind = len(tiffs)
    da = xr.concat([rxr.open_rasterio(Path(tiff_dir,f)).squeeze(dim='band') for f in tiffs[:stop_ind]], dim=times[:stop_ind])
    da = da.drop_vars(['band', 'spatial_ref'])

    print(f"Dataset size in memory: {da.nbytes / 1e6:.2f} MB")

    return da, tiffs, times

In [ ]:
### --- Load data --- ###
# da_tot - Xarray containing all of the data. 
# tiffPaths_tot - posix paths array containing paths to all data files. 
# times_tot - list containing all of the dates of each data (note: this will also be in 'da_tot').

start_time = time.time()
type_file = 'SAR'

# # Load VV tiffs
pola = 'VV' 
da_VV_tot, tiffPaths_VV_tot, times_VV_tot = load_tiffs(fc.selected, type_file, pola, stop_ind = -1)

# Load VH tiffs
pola = 'VH'
da_VH_tot, tiffPaths_VH_tot, times_VH_tot = load_tiffs(fc.selected, type_file, pola, stop_ind = -1)

end_time = time.time()

print(f'Data load took {end_time - start_time:.2f} seconds.')

Remove fill values. 

For data in decibels, this was found to be the smallest value. 

In [ ]:
# Replace fill values with NaN. 
replaceThis = da_VH_tot[0]._FillValue
da_VH_tot = da_VH_tot.where(da_VH_tot != replaceThis, other=np.nan)
da_VV_tot = da_VV_tot.where(da_VV_tot != replaceThis, other=np.nan)

## Apply Median Filter

Apply a median filter to the data. This is often a standard step when dealing with SAR data, which is inherently noisy. 

In [ ]:
for i in range(len(da_VV_tot)):
    da_VV_tot[i].values = signal.medfilt2d(da_VV_tot[i].values, kernel_size = median_size)

In [ ]:
for i in range(len(da_VH_tot)):
    da_VH_tot[i].values = signal.medfilt2d(da_VH_tot[i].values, kernel_size = median_size)

***
# 3. Examine SAR Imagery

This section simply plots the raw SAR imagery. Data is assumed to be in decibels. 

In [ ]:
def subplot_SAR_in_dB(da_tot, vmin=-25, vmax=-5): 

    num_modes = len(da_tot)

    # Create subplots
    n_cols = 2
    n_rows = int(np.ceil(num_modes/2))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(12,n_rows*4))

    # Flatten axes array for easier indexing. 
    axes = axes.flatten()

    # Loop through and plot the modes. 
    for i in range(num_modes): 
        ax = axes[i]
        # myData_dB = 20 * np.log10(np.abs(da_tot[i].copy(deep=True).values + 1e-10))
        # ax.imshow(myData_dB, cmap='gray_r', vmin=vmin, vmax=vmax, interpolation=None)
        # ax.imshow(abs(da_tot[i].values), cmap='gray', vmin=vmin, vmax=vmax, interpolation=None)
        ax.imshow(da_tot[i].values, cmap='gray', vmin=vmin, vmax=vmax, interpolation=None)
        ax.set_title(f"Date: {da_tot[i].time.values.astype('datetime64[D]')}")

    # Turn off unused subplots
    for i in range(num_modes, len(axes)): 
        ax = axes[i]
        ax.set_axis_off()

    # Adjust layout
    plt.tight_layout()

    # Show plot. 
    plt.show()
    
    return

In [ ]:
subplot_SAR_in_dB(da_VH_tot, vmin=-35, vmax=-5)

In [ ]:
subplot_SAR_in_dB(da_VV_tot, vmin=-30, vmax=-5)

***
# 4. Calculate Change in dB levels

In [ ]:
# Initialize array to contain differences. 
t, y, x = da_VH_tot.shape[0]-1, da_VH_tot.shape[1], da_VH_tot.shape[2]
da_diffs_VH = np.zeros((t,y,x))
t, y, x = da_VV_tot.shape[0]-1, da_VV_tot.shape[1], da_VV_tot.shape[2]
da_diffs_VV = np.zeros((t,y,x))

In [ ]:
def daDiffCalc(da_tot, w_s):
    # Function to get the difference between consecutive days in a data array. 
    # Note: This uses a median filter to stabilize values. 
    
    # da_tot - xarray data to get difference between consecutive days. 
    # w_s - window size to consider. 
    
    ts, ys, xs = da_tot.shape
    da_diffs = np.zeros((ts-1,ys,xs))
    temp = np.zeros((ys, xs))

    for i in range(ts-1): 
        temp = da_tot[i+1].values - da_tot[i].values
        da_diffs[i] = signal.medfilt2d(temp, kernel_size=w_s)

    return da_diffs

In [ ]:
### --- Calculate difference between consecutive days --- ###

# Initialize container for differences. 
ts, ys, xs = da_VH_tot.shape
da_diffs_VH = np.zeros((len(window_sizes),ts-1,ys,xs))
da_diffs_VV = np.zeros((len(window_sizes),ts-1,ys,xs))

# Calculate differences. 
start_time = time.time()
for i, w_s in enumerate(window_sizes):
    # Run function to get differences. 
    da_diffs_VH[i] = daDiffCalc(da_VH_tot, w_s)
    da_diffs_VV[i] = daDiffCalc(da_VV_tot, w_s)

end_time = time.time()

print(f'It took {end_time-start_time:.2f} seconds to calculate decibel changes between consecutive days. ')

# 5. Find Overflow

Overflow seems to be characterized by a small drop in backscatter following consistent increases. The below code finds possible overflow eventws by searching for positive changes in backscatter (i.e., the backscatter is increasing) followed by a negative change in backscatter (i.e., the backscatter decreases). 

There are three sections:
* 5.1 - Code to find and display Overflow
  * This section sets up all of the code to find, cull, and display the overflow results. 
* 5.2 - Run code to find overflow
  * This section runs the code to find, cull, and display the overflow results.
  * This will save the results as one or more geotiffs. 

## 5.1 Code to find and display overflow

The below code finds which individual pixels meet the criteria for possible overflow. 

In [ ]:
def findOverflowIndices(da_diffs, pos = 0.5, neg = -0.5, start_ind = 0, stop_ind = -1):
    # Function to find indices of overflow. 
    # This method is predicated on overflow being preceded by increasing backscatter, and then decreasing backscatter. 
    # Note: This automatically filters by time. 

    # Input variables:
    # da_diffs - array containing differences. 
    # pos - minimum value for positives. 
    # neg - minimum value for negatives. 

    # Output variables: 
    # myIndices - list of all indices for possible overflow. 
    # idx0 - x indices of myIndices as a list; this can be easier to use than myIndices. 
    # idx1 - y indices of myIndices as a list. 

    # initialize some containers
    list_pos = []
    list_neg = []

    if stop_ind < 0:
        stop_ind = len(da_diffs) + stop_ind + 1
    if start_ind < 0:
        start_ind = len(da_diffs) + start_ind + 1

    # print(f"start_ind = {start_ind}\nstop_ind  = {stop_ind}")

    # Loop through all of the difference pairs. 
    for i in range(start_ind-1, stop_ind):
        # get temporary list of positive "nows" and negative "tomorrows"
        temp_pos = np.where(da_diffs[i-1] > pos)
        temp_neg = np.where(da_diffs[i] < neg)

        # Put positives and their date indices into a list. 
        for j in range(len(temp_pos[0])):
            list_pos.append([temp_pos[0][j], temp_pos[1][j], i])

        # Put negatives and their date indices into a list. 
        for j in range(len(temp_neg[0])):
            list_neg.append([temp_neg[0][j], temp_neg[1][j], i])

    # Put the indices into a single list if they have the same location and time index. 
    myIndices = [list(item) for item in set(map(tuple, list_pos)) & set(map(tuple, list_neg))]
    
    idx0 = [sublist[0] for sublist in myIndices]
    idx1 = [sublist[1] for sublist in myIndices]
    
    return myIndices, idx0, idx1

In [ ]:
def removeDups(myList, myListWithDups):
    # Function to remove duplicate lists from a list of lists found in a different list of lists. 
    # myListWithDups - list of lists with duplicate lists to be removed. 
    # myList - list of duplicates to be removed from myListWithDups. 

    # Convert lists in myList to a set of tuples for efficient lookup. 
    set_myList = set(tuple(sub) for sub in myList)

    # Use list comprehension to keep items in myListWithDups that are not in set_myList
    myListWithoutDups = [sub for sub in myListWithDups if tuple(sub) not in set_myList]
    
    return myListWithoutDups

In [ ]:
### Create plotting function for possible overflow! ###
def plotOverflowProbabilities(da, myIndices, threshs, cmap = 'viridis'):
    # Function to plot probable overflow locations. 
    # da - single SAR image. 
    # myIndices - list of lists of lists; this contains the indices for each identified possible overflow point for different

    # plot SAR image background. 
    fig = plt.figure(figsize=(12,9))
    plt.imshow(da, cmap='gray', interpolation='None')

    # Create color list based on number of myIndices lists. 
    n_colors = len(myIndices)
    color = cm.rainbow(np.linspace(0, 1, n_colors))

    # Loop through each list of myIndices. 
    for i in range(len(myIndices)-1, -1, -1):
        # get necessary indices.     
        idx0 = [sublist[0] for sublist in myIndices[i]]
        idx1 = [sublist[1] for sublist in myIndices[i]]        

        plt.plot(idx1, idx0, color=color[i], marker='.', markersize=1, linestyle='None', alpha=0.5, label=f'thresh={threshs[i]}')

    plt.title("Raw Overflow")
    plt.legend(markerscale=10.0)
    # Add grid lines. 
    plt.grid()
    # Add title. 
    plt.show()
    
    return

In [ ]:
def collectMyIndices(da_diffs_VH, da_diffs_VV, min_pos, min_neg, start_ind, stop_ind):
    # Function to get all overflow indices. 

    start_time = time.time()
    ### --- Collect VH indices. --- ###
    myIndices_VH = []
    # Get list of indices for all probabilities (i.e., thresholds). 
    for neg in min_neg:
        myIndices_temp, _, _ = findOverflowIndices(da_diffs_VH[0], pos=min_pos, neg=neg, start_ind=start_ind, stop_ind=stop_ind)
        myIndices_VH.append(myIndices_temp)    
    # Remove duplicates from lower probability lists. 
    if len(min_neg) > 1:
        for i in range(len(min_neg)-1):
            myIndices_VH[i+1] = removeDups(myIndices_VH[i], myIndices_VH[i+1])
    for i in range(len(myIndices_VH)):
        print(f"For threshold = {min_neg[i]}, there are {len(myIndices_VH[i])} possible overflow pixels identified using VH only.")

    ### --- Collect VV indices. --- ###
    myIndices_VV = []
    # Get list of indices for all probabilities (i.e., thresholds)
    for neg in min_neg:
        myIndices_temp, _, _ = findOverflowIndices(da_diffs_VV[0], pos=min_pos, neg=neg, start_ind=start_ind, stop_ind=stop_ind)
        myIndices_VV.append(myIndices_temp)
    # Remove duplicates from lower probability lists. 
    if len(min_neg) > 1:
        for i in range(len(min_neg)-1):
            myIndices_VV[i+1] = removeDups(myIndices_VV[i], myIndices_VV[i+1])
    for i in range(len(myIndices_VH)):
        print(f"For threshold = {min_neg[i]}, there are {len(myIndices_VV[i])} possible overflow pixels identified using VV only.")

    ### --- Combine VV & VH indices. --- ###
    myIndices_both = []
    for i in range(len(myIndices_VV)): 
        myIndices_temp = [list(item) for item in set(map(tuple, myIndices_VH[i])) & set(map(tuple, myIndices_VV[i]))]
        myIndices_both.append(myIndices_temp)
        print(f"For threshold = {min_neg[i]}, there are now {len(myIndices_both[i])} possible overflow pixels identified using both VV & VH.")
        
    end_time = time.time()

    print(f'It took {end_time - start_time:.2f} seconds to collect the indices.')

    ### --- Plot combined results only. --- ###
    plotOverflowProbabilities(da_VV_tot[0], myIndices_both, min_neg)


    return myIndices_both

In [ ]:
def groupAllByContact(myIndices, min_neg, da_tot, min_pixels=4, max_pixels=float('inf')):
    # Function to group overflow locations into a dictionary for easy calling/use. 

    # Input variables: 
    # myIndices - list of indices that have been flagged as possible overflow. If len(min_neg) > 1, this will be a list of lists. 
    # min_neg - list of negative thresholds used to determine if a pixel is considered overflow. This can be a single number. 
    # da_tot - xarray containing the original SAR image data. 
    # min_pixels - minimum number of pixels a group needs in order to be retained. 

    # Output variables: 
    # myDates - list of dates as strings. 
    # myDatas - raw and statistical data as a dictionary. 

    # Initialize dictionary container for the data. 
    myDatas = {}
    # Initialize container for indices. 
    myIndices_grouped = []

    for i in range(len(min_neg)): 
        print(f'  Starting grouping for threshold={min_neg[i]} with {len(myIndices[i])} pixels.')
        start_time = time.time()
        # Get indices from myIndices
        if len(min_neg) > 1:
            idx0 = [sublist[0] for sublist in myIndices[i]]
            idx1 = [sublist[1] for sublist in myIndices[i]]
        else: 
            idx0 = [sublist[0] for sublist in myIndices]
            idx1 = [sublist[1] for sublist in myIndices]

        # Create a binary mask of overflow pixels
        ts, ys, xs = da_tot.shape # get the shape needed. 
        overflow_ungrouped = np.zeros((ys,xs)) # initialize as an array of zeros. 
        overflow_ungrouped[idx0, idx1] = 1 # add 1's for indices that have been identified as possible overflow. 

        # Call code to group each set of pixels. 
        myDatas[f'thresh={min_neg[i]}'], myDates, myIndices_temp = groupOneByContact(overflow_ungrouped, da_tot, myIndices, min_pixels=min_pixels, max_pixels=max_pixels)
        myIndices_grouped.append(myIndices_temp)

        end_time = time.time()
        print(f'  It took {end_time-start_time:.2f} seconds to group the pixels for threshold={min_neg[i]}. ')
        
    
    return myDatas, myDates, myIndices_grouped

def groupOneByContact(possible_overflow, da_tot, myIndices, min_pixels=4, max_pixels=float('inf')):
    # Function to group overflow locations into a dictionary for easy calling/use. 

    # Input variables: 
    # possible_overflow - the mask of potential overflow (0 where no overflow; 1 where possibly detected)
    # da_tot - xarray containing the original SAR image data. 
    # min_pixels - minimum number of pixels a group needs in order to be retained. 

    # Output variables: 
    # myDates - list of dates as strings. 
    # myData  - raw and statistical data as a dictionary. 

    
    # Group locations by pixels that touch. 
    # connectivity = 1 for only up/down connections. 
    # connectivity = 2 for up/down and diagonal connections. 
    labeled_image = label(possible_overflow, connectivity=2)

    # Remove groups of pixels that are smaller than 'min_pixels.' 
    for i in range(1,np.max(labeled_image)+1): 
        pixelIndices = np.where(labeled_image == i)
        pixelCount = len(pixelIndices[0])
        if (pixelCount < min_pixels) or (pixelCount > max_pixels): 
            labeled_image[np.where(labeled_image == i)] = 0

    # add indices to 'myIndices_grouped' that have connectivity. 
    myIndices_grouped = []
    temp = np.where(labeled_image > 0)
    for i in range(len(temp[0])):
        myIndices_grouped.append([temp[0][i], temp[1][i]])

    # Put raw data into dictionary for each grouping. 
    # Get unique entries and their counts. Used for organizing later. 
    unique_vals, counts = np.unique(labeled_image, return_counts=True)    
    # initialize container for backscatter values. 
    myData = {}    
    myDates = []
    for val in unique_vals[1:]: # skipping 0 since that will just be non-overflow. 
        myIndex = np.where(labeled_image == val)
        ts = len(da_tot)
        ps = len(myIndex[0])
        myDates = da_tot[:].time.values.astype('datetime64[D]')
        myValues = np.zeros(( ts, ps))
        # Add values to dictionary. 
        myD = {}
        for i in range(ps):
            myValues[:, i] = da_tot[:, myIndex[0][i], myIndex[1][i]].values
        myD['raw'] = myValues
        myData[f"{val}"] = myD
        # Add indices of location to dictionary. 
        myIndices = []
        for i in range(len(myIndex[0])):
            myIndices.append([myIndex[0][i], myIndex[1][i]])
        myData[f"{val}"]['indices'] = myIndices

    # Get statistics into 'myDatas.' 
    for key in myData:
        # Initialize containers for mean, median, and std. 
        myMeans   = np.zeros((1,0))
        myMedians = np.zeros((1,0))
        myStds    = np.zeros((1,0))
        # Get stats for each key. 
        for i in range(len(myData[key]['raw'])):
            myMean =  np.nanmean(myData[key]['raw'][i])
            myMeans = np.append(myMeans,myMean)
    
            myMedian =  np.nanmedian(myData[key]['raw'][i])
            myMedians = np.append(myMedians,myMedian)
    
            myStd =  np.nanstd(myData[key]['raw'][i])
            myStds = np.append(myStds, myStd)
        
    
        # Put stats in dictionary. 
        myData[key]['mean']   = myMeans
        myData[key]['median'] = myMedians
        myData[key]['std']    = myStds
    
    return myData, myDates, myIndices_grouped

In [ ]:
# Plot data for user to look through graphs and identify likely days/locations with overflow. 

def plotdBForEachLocation(myDatas, myDates, ymin=-24.0, ymax=-5.0, start_ind=0, stop_ind=-1):

    # loop through each key in 'myDatas' dictionary. 
    # Note: The keys should correspond to the unique values of possible overflow groupings. 

    if len(myDatas) == 0:
        print("No data to plot.") 
    else: 
        ncols = 2
        nrows = int(np.ceil(len(myDatas)/2))
    
        n_max = 20
        if nrows > n_max:
            print(f"Too many individual indices.\nPlotting only those from {start_ind} to {start_ind+20}.")
            print("To plot a different range of individual points, set 'start_ind' and 'stop_ind'.")
            nrows = int(n_max / 2)
            stop_ind = start_ind + n_max - 1
        else: 
            stop_ind = len(myDatas)
        
        fig_find, axes_find = plt.subplots(nrows=nrows, ncols=ncols, figsize=(9,nrows*4))
        
        axes_find = axes_find.flatten()
        
        for i, key in enumerate(myDatas): 
            if i >= start_ind and i <= stop_ind:
                ax = axes_find[i]
                ax.plot(myDates, myDatas[key]['mean'], color='blue')
                ax.plot(myDates, myDatas[key]['mean']+myDatas[key]['std'], color='blue', marker='v', linestyle='None')
                ax.plot(myDates, myDatas[key]['mean']-myDatas[key]['std'], color='blue', marker='^', linestyle='None')
                ax.plot(myDates, myDatas[key]['median'], color='green')
                ax.grid()    
                ax.set_xticks(myDates, myDates, rotation=45)
                ax.set_title(f"Location Number: {key}")
                ax.set_ylim(ymin, ymax)
    
        for j in range(i+1, len(axes_find)):
            ax = axes_find[j]
            ax.set_axis_off()
        
        plt.tight_layout()
        plt.show()

    return

In [ ]:
### Create plotting function for possible overflow! ###
def plotGroupedOverflowProbabilities(da, myDatas, min_neg, cmap = 'viridis'):
    # Function to plot probable overflow locations. 
    # da - single SAR image. 
    # myDatas - dictionary containing the data. 
    # myIndices - list of lists of lists; this contains the indices for each identified possible overflow point for different

    # plot SAR image background. 

    # Create color list based on number of myIndices lists. 
    n_colors = len(min_neg)
    color = cm.rainbow(np.linspace(0, 1, n_colors))
        
    # Loop through the dictionary
    i = 0
    for thresh in myDatas:
        fig = plt.figure(figsize=(12,9))
        plt.imshow(da, cmap='gray', interpolation='None')
        for key in myDatas[thresh]: 
            idx1_temp, idx0_temp = [], []
            for j in range(len(myDatas[thresh][key]['indices'])):
                idx1_temp.append(myDatas[thresh][key]['indices'][j][1])
                idx0_temp.append(myDatas[thresh][key]['indices'][j][0])
            plt.plot(idx1_temp, idx0_temp, color=color[i], marker='.', markersize=1, linestyle='None')
            plt.annotate(f"{key}", xy=(int(np.min(idx1_temp))+10, int(np.max(idx0_temp)-10)))
        
        i = i + 1

        # Add grid lines. 
        plt.title(f"Grouped Overflow\n{thresh}")
        plt.grid()
        plt.show()
    
    return

In [ ]:
def groupMyIndices(myIndices, min_neg, da_tot, min_pixels, max_pixels):
    # Function to group all of the indices. 

    ### --- Group indices. --- ###
    start_time = time.time()
    myDatas, myDates, myIndices_grouped = groupAllByContact(myIndices, 
                                            min_neg, 
                                            da_tot, 
                                            min_pixels=min_pixels,
                                            max_pixels=max_pixels)
    end_time = time.time()
    print(f'It took {end_time-start_time:.2f} seconds to group the pixels. ')

    ### --- Plot grouped indices. --- ###
    plotGroupedOverflowProbabilities(da_VV_tot[0], myDatas, min_neg)

    return myDatas, myDates, myIndices_grouped

### Code to save overflow as a geotiff

In [ ]:
def saveOverflowAsGeotiff(loadLoc, saveLoc, da, min_neg, myIndices, saveName = None):
    # Save overflow as a geotiff

    # Get important GIS info. 
    tiff_dir = Path(loadLoc.selected,'RTC_GAMMA/')
    tiffs = list(tiff_dir.glob(f'*.tif*'))
    raster = rasterio.open(tiffs[0])
    CRS = raster.crs
    transform = raster.transform
    raster.close()

    # Set up empty array
    myGeotiffArray = np.zeros(da.shape, dtype=np.int8)

    # Walk through indices; this goes through reverse. 
    j = 0 # set up a counter that increases. Higher number means higher likelihood of flooding. 
    if len(min_neg) > 1: 
        for i in range(len(myIndices)-1, -1, -1):
            j = j + 1
            idx0 = [sublist[0] for sublist in myIndices[i]]
            idx1 = [sublist[1] for sublist in myIndices[i]]            
            myGeotiffArray[idx0,idx1] = int(j)
    else: 
        idx0 = [sublist[0] for sublist in myIndices]
        idx1 = [sublist[1] for sublist in myIndices]
        myGeotiffArray[idx0,idx1] = int(1)

    # Set up save settings. 
    profile = {
        'driver': 'GTiff',
        'height': myGeotiffArray.shape[0],
        'width': myGeotiffArray.shape[1],
        'count': 1,
        'dtype': myGeotiffArray.dtype,
        'crs': CRS,
        'transform': transform
    }
    
    # Write data
    if not saveName:
        saveName = Path(saveLoc.selected,'overflow_'+str(date.today())+'.tif')
    else: 
        saveName = Path(saveLoc.selected,saveName+'.tif')
    with rasterio.open(saveName, 'w+', **profile, compress='lzw') as dst:
        dst.write(myGeotiffArray, 1)
    
    return myGeotiffArray

## 5.2 Run Code to find overflow

### <span style="color:red"><b>USER INPUT MODIFICATION CAN BE MADE HERE!!!</b></span>

min_pos - smallest positive change prior to the drop in backscatter. <br>
min_neg - smallest negative change to backscatter. 

start_ind - starting index at which to look for overflow. <br>
stop_ind  - last index for which to look for overflow. Setting this to -1 will go to the last index. 

To modify the user input, delete the '#' symbol and change the value to desired setting. 

In [ ]:

print(f"min_pos = {min_pos}")
print(f"min_neg = {min_neg}\n")
print(f"start_ind = {start_ind}")
print(f"stop_ind  = {stop_ind}")
print(f"\nIndividual or Aggregate = {aggOrInd}")
print(f"\nsaveName = {saveName}")
print(f"\nmin_pixels = {min_pixels}")
print(f"max_pixels = {max_pixels}")

In [ ]:
# set the minimum preceding increase and the minimum decrease in backscatter to flag as overflow. 
# min_pos = -0.1
min_neg = [-5.0, -3.0, -1.0]
min_neg.sort()

# Set start and stop indices. This is useful if you are dealing with e.g., several months of data that includes time that can't possibly have overflow. 
# start_ind = 0
# stop_ind = -1

# aggOrInd = 'individual'
# aggOrInd = 'aggregate'

# saveName = None
# saveName = 'multiOverflowTest'

# # Typical overflow event are approximately 15 S1 pixels in size. 
min_pixels = 1
# max_pixels = float('inf')

In [ ]:
# make negative indices machine understandable. 
if start_ind < 0: 
    start_ind = len(da_VV_tot) + start_ind
    print(f"New start_ind = {start_ind}")
if stop_ind < 0:
    stop_ind = len(da_VV_tot) + stop_ind
    print(f"New stop_ind = {stop_ind}")

# Initialize container for plotting amount of overflow detected through time. 
myIndices_thruTime = []

if 'aggregate' == aggOrInd:
    # find overflow. 
    myIndices = collectMyIndices(da_diffs_VH, da_diffs_VV, min_pos, min_neg, start_ind, stop_ind)
    # group overflow. 
    if min_pixels > 1: 
        myDatas, myDates, myIndices_grouped = groupMyIndices(myIndices, min_neg, da_VV_tot, min_pixels, max_pixels)
        myIndices = myIndices_grouped
    # save overflow. 
    sN = saveName+'_'+str(da_VV_tot[start_ind].time.values.astype('datetime64[D]'))+'_'+str(da_VV_tot[stop_ind].time.values.astype('datetime64[D]'))
    print(f"Aggregate saveName = {sN}")
    _ = saveOverflowAsGeotifftiff(fc, fc_saveLoc, da_VV_tot[0], min_neg, myIndices, saveName = sN)
elif 'individual' == aggOrInd:
    # loop through each consecutive pair. 
    for i in range(start_ind, stop_ind):
        # find overflow. 
        myIndices = collectMyIndices(da_diffs_VH, da_diffs_VV, min_pos, min_neg, i, i+1)
        myIndices_thruTime.append(myIndices)
        # group overflow. 
        if min_pixels > 1:
            myDatas, myDates, myIndices_grouped = groupMyIndices(myIndices, min_neg, da_VV_tot, min_pixels, max_pixels)
            myIndices = myIndices_grouped
        # save overflow. 
        sN = saveName+'_'+str(da_VV_tot[i].time.values.astype('datetime64[D]'))+'_'+str(da_VV_tot[i+1].time.values.astype('datetime64[D]'))
        print(f"individual saveName = {sN}")
        _ = saveOverflowAsGeotiff(fc, fc_saveLoc, da_VV_tot[0], min_neg, myIndices, saveName = sN)

## 5.3 Plot amount of overflow detected through time

In [ ]:
def plotOverflowCountThruTime(myIndices_thruTime, da_VV_tot, threshs): 
    # Plot the number of overflow counts through time. 

    ### --- Get values for plotting. --- ###
    dummyXs = list(range(0,len(myIndices_thruTime)))
    Ys = np.zeros((len(dummyXs),len(threshs)))
    for i in range(len(myIndices_thruTime)):
        for j in range(len(threshs)): 
            Ys[i,j] = len(myIndices_thruTime[i][j])
                  
        

    ### --- Plot the amount of overflow through time. --- ###
    # Create color list based on number of thresholds. 
    n_colors = len(threshs)
    colors = cm.rainbow(np.linspace(0,1,n_colors))

    # Create the figure and plot on it. 
    fig_OFTT = plt.figure()
    for j in range(len(min_neg)): 
        plt.plot(dummyXs, Ys[:,j], color=colors[j], marker='.', markersize=1, label=f'thresh={threshs[j]}')

    # Create new xtick labels. 
    myXTicks = []
    for i in range(len(da_VV_tot)-1):
        myXTicks.append(f'{da_VV_tot[i].time.values.astype('datetime64[D]')}\n    &\n{da_VV_tot[i+1].time.values.astype('datetime64[D]')}')
    plt.xticks(ticks=dummyXs, labels=myXTicks, rotation=45)

    
    # Make the graph pretty. 
    plt.title("Number of Pixels Identified as Overflow Through Time")
    plt.legend(markerscale = 10.0)
    plt.grid()
    plt.tight_layout()
    plt.show()


    return

In [ ]:
def plotOverflowCountThruTime_v2(myIndices_thruTime, da_VV_tot, threshs, figsize=(12,4)): 
    # Plot the number of overflow counts through time. 

    ### --- Get values for plotting. --- ###
    dummyXs = list(range(0,len(myIndices_thruTime)))
    Ys = np.zeros((len(dummyXs),len(threshs)))
    for i in range(len(myIndices_thruTime)):
        for j in range(len(threshs)): 
            Ys[i,j] = len(myIndices_thruTime[i][j])

    ### --- Plot the amount of overflow through time. --- ###
    # Create color list based on number of thresholds. 
    n_colors = len(threshs)
    colors = cm.rainbow(np.linspace(0,1,n_colors))
    
    # Create new xtick labels. 
    myXTicks = []
    for i in range(len(da_VV_tot)-1):
        myXTicks.append(f'{da_VV_tot[i].time.values.astype('datetime64[D]')}\n    &\n{da_VV_tot[i+1].time.values.astype('datetime64[D]')}')

    # Create the figure and plot on it. 
    fig_OFTT, axes_OFTT = plt.subplots(nrows=len(threshs), ncols=1, figsize=figsize)
    axes_OFTT = axes_OFTT.flatten()
    for j in range(len(min_neg)):#-1,-1,-1): 
        print(j)
        axes_OFTT[j].plot(dummyXs, Ys[:,j], color=colors[j], marker='.', markersize=1)
        
        axes_OFTT[j].grid()
        if j == len(min_neg):
            axes_OFTT[j].set_xticks(dummyXs, myXTicks,rotation=45)
        else: 
            axes_OFTT[j].set_xticks
        axes_OFTT[j].set_title(f"Thresh = {threshs[j]}")
    
    # Make the graph pretty. 
    plt.suptitle("Number of Pixels Identified as Overflow Through Time")
    plt.tight_layout()
    plt.show()


    return

In [ ]:
plotOverflowCountThruTime(myIndices_thruTime, da_VV_tot, min_neg)

In [ ]:
plotOverflowCountThruTime_v2(myIndices_thruTime, da_VV_tot, min_neg, figsize=(8,16))